# Sprint E4 walkthrough: PCA, the covariance lab and the residual audit

Statistical factor models and the covariance laboratory. This notebook reads
artifacts and asserts every number it prints against the stored value; nothing
here is typed by hand. The last cell scans this file's own code cells for any
stored number appearing as a literal and fails if it finds one.

Inputs: `data/processed/returns.parquet`, `data/processed/sectors.parquet`,
`data/models/PCA-v1/*`, `data/models/PCA-v1c/*`, `data/models/registry.json`,
`data/eval/cov_horse_race.parquet`, `data/eval/e4_f41_pc1_correlations.parquet`,
`data/eval/e4_f44_held_out.parquet`, `data/eval/xs_residual_spectrum.parquet`,
`data/eval/xs_task3_*.parquet`, `data/eval/xs_survivor_*.parquet`,
`sprints/E4/RESULTS.json`.

Output: none. The notebook prints and asserts; the artifacts are written by
`make rebuild-e4` and the criteria by `efb/evaluate.py`.

In [1]:
# the repository root is importable so the package and the dashboard module can
# be imported from the notebook's working directory
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from efb import cov, evaluate, registry, tercile  # noqa: E402
from efb.models import statistical as st  # noqa: E402
from efb import pca_eval  # noqa: E402

DATA = ROOT / "data"
RESULTS = json.loads((ROOT / "sprints" / "E4" / "RESULTS.json").read_text())
REGISTRY = json.loads((DATA / "models" / "registry.json").read_text())
STORED = {name: block["stored_numbers"] for name, block in RESULTS["criteria"].items()}
VERDICTS = {name: block["verdict"] for name, block in RESULTS["criteria"].items()}
print("criteria loaded:", sorted(STORED))
print("versions registered:", sorted(REGISTRY["models"]))

criteria loaded: ['F4.1', 'F4.2', 'F4.3', 'F4.4', 'F4.5', 'F4.6', 'F4.7']
versions registered: ['PCA-v1', 'PCA-v1c', 'TS-v1', 'XS-v1']


In [2]:
# Cell 1 rule: the data hash in the results file must be the hash of the
# artifacts this notebook reads, recomputed here from the files on disk.
recomputed = evaluate.e4_data_hash(DATA)
stored_hash = str(RESULTS["data_hash"])
print("data_hash   ", stored_hash)
print("recomputed  ", recomputed)
assert stored_hash == recomputed, "the results file and the artifacts disagree"
assert stored_hash.startswith("ff4e6b2d0002ab65"), "the expected sprint hash moved"
for path in evaluate.e4_artifacts(DATA):
    assert path.exists(), path
print(f"{len(evaluate.e4_artifacts(DATA))} artifacts hashed")

data_hash    ff4e6b2d0002ab65e7e905e0908f57258236419454cfbb904b8cc85a47679018
recomputed   ff4e6b2d0002ab65e7e905e0908f57258236419454cfbb904b8cc85a47679018
16 artifacts hashed


## 1. Every criterion, its stored number and its verdict

F4.1 to F4.4 are copied verbatim from `docs/roadmap_v2.md`. F4.5 and F4.6 took
their own IDs in `sprints/E4/PRD.md` when Tasks 3 and 4 were reframed. F4.7 was
registered mid-sprint by decision and is recorded as such.

INPUT: `sprints/E4/RESULTS.json`.
OUTPUT: none, printed and asserted only.

In [3]:
# the criterion text is printed as stored, so a reworded threshold would be
# visible in this notebook rather than only in the results file
for name in [f"F4.{index}" for index in range(1, 8)]:
    block = RESULTS["criteria"][name]
    print(f"{name}  {block['verdict']:4s}  {block['threshold']}")
    print(f"      {block['criterion']}")
assert sorted(RESULTS["criteria"]) == [f"F4.{index}" for index in range(1, 8)]
assert set(VERDICTS.values()) == {"pass", "fail"}
print()
print("pass:", sorted(k for k, v in VERDICTS.items() if v == "pass"))
print("fail:", sorted(k for k, v in VERDICTS.items() if v == "fail"))
assert VERDICTS["F4.1"] == "fail" and VERDICTS["F4.4"] == "fail"

F4.1  fail  correlation > 0.95
      First principal component vs the market factor: correlation above 0.95.
F4.2  pass  3 <= factor count <= 15, the STOP CONDITION 3 band
      Marchenko-Pastur edge isolates between 3 and 15 significant factors; the count is stored.
F4.3  pass  every shrinkage and factor estimator's median at least 10% below the sample covariance's median
      OOS minimum-variance vol: shrinkage and factor estimators beat the sample covariance by more than 10%. If the sample covariance wins, that is an estimation bug.
F4.4  fail  PCA-v1 held-out R squared >= XS-v1 held-out R squared, both stored
      PCA-v1 with k factors explains at least as much cross-sectional variance as XS-v1 on a held-out residual test; both numbers stored.
F4.5  pass  3a, 3b, 3c and 3d measured and the explanation named
      Task 3 as reframed: passes when 3a, 3b, 3c and 3d are measured and the deliverable names which explanation the evidence supports.
F4.6  pass  4a, 4b and 4c stored
      

In [4]:
# the headline number of each criterion, read from the stored block: no value
# below is written in this cell, only the path to it
headlines = {
    "F4.1": STORED["F4.1"]["pc1_vs_market_pca_v1"],
    "F4.2": STORED["F4.2"]["n_factors_mp"],
    "F4.3": STORED["F4.3"]["worst_ratio"],
    "F4.4": STORED["F4.4"]["pca_rolling_refit"],
    "F4.5": STORED["F4.5"]["momentum_share_of_predicted_variance"]["high"],
    "F4.6": STORED["F4.6"]["style_correlation_panel_vs_mapped"]["size"],
    "F4.7": STORED["F4.7"]["pca_v1c_n_factors_above_edge"],
}
for name, value in headlines.items():
    print(f"{name}  {value}")
assert 3 <= headlines["F4.2"] <= 15, "STOP CONDITION 3 band"
assert headlines["F4.3"] < 0.9
assert headlines["F4.6"] < 0.9

F4.1  0.7970186544462351
F4.2  13
F4.3  0.5715118239875542
F4.4  0.31330323475183786
F4.5  0.5612671526099806
F4.6  0.5777797124005946
F4.7  16


## 2. The residual structure as one object, four measurements

The through-line of the sprint. Four numbers, read from four artifacts, are
four views of one fact: XS-v1's specific returns are not independent, and the
diagonal specification of `D` cannot express that.

INPUT: `data/eval/xs_residual_spectrum.parquet`,
`data/eval/xs_task3_projection.parquet`, `data/eval/xs_task3_orthogonality.parquet`,
`data/eval/e4_f44_held_out.parquet`, `data/models/registry.json`.
OUTPUT: none, printed and asserted only.

In [5]:
spectrum = pd.read_parquet(DATA / "eval" / "xs_residual_spectrum.parquet")
projection = pd.read_parquet(DATA / "eval" / "xs_task3_projection.parquet")
orthogonality = pd.read_parquet(DATA / "eval" / "xs_task3_orthogonality.parquet")
held_out = pd.read_parquet(DATA / "eval" / "e4_f44_held_out.parquet").set_index("row")

largest = float(spectrum.loc[spectrum["index"] == 1, "eigenvalue"].max())
edge = float(spectrum.loc[spectrum["index"] == 1, "mp_edge"].max())
above = int(spectrum["above_edge"].sum())
print(f"1. largest residual eigenvalue {largest:.4f} against an MP edge of {edge:.4f}, "
      f"{above} above the edge")

by_tercile = projection.groupby("tercile")[["top5_share", "effective_directions", "n_names"]].mean()
print("2. share of the book's specific variance in the top five residual directions")
print(by_tercile["top5_share"].to_string())
print(f"   across {by_tercile['n_names'].mean():.1f} names and "
      f"{by_tercile['effective_directions'].mean():.1f} effective directions")

component_covariance = orthogonality.groupby("tercile")["covariance"].mean()
print("3. realized covariance between the factor and specific components")
print(component_covariance.to_string())

gain_three = float(held_out.loc["(v) XS-v1 plus top 3 residual PCs", "mean_r_squared"])
frozen = float(held_out.loc["(ii) XS-v1 descriptors frozen", "mean_r_squared"])
print(f"4. held-out R squared {frozen:.6f} frozen, {gain_three:.6f} with three "
      f"residual PCs, a gain of {(gain_three - frozen) * 100:.1f} points")

assert largest > edge
assert above > 0
assert by_tercile["top5_share"].min() > 0.55 and by_tercile["top5_share"].max() < 0.58
assert component_covariance["high"] < 0 and component_covariance["low"] > 0
assert gain_three > frozen
print()
print("all four measurements are the same finding: the specific component has")
print("common directions a diagonal cannot represent, and adding them back is")
print("worth more than any descriptor change measured this sprint.")

1. largest residual eigenvalue 22.3031 against an MP edge of 3.9522, 23 above the edge
2. share of the book's specific variance in the top five residual directions
tercile
high    0.576803
low     0.568137
mid     0.552868
   across 151.3 names and 11.4 effective directions
3. realized covariance between the factor and specific components
tercile
high   -1.039161e-06
low     1.290002e-06
mid     7.352554e-07
4. held-out R squared 0.253997 frozen, 0.292699 with three residual PCs, a gain of 3.9 points

all four measurements are the same finding: the specific component has
common directions a diagonal cannot represent, and adding them back is
worth more than any descriptor change measured this sprint.


## 3. By hand: the eigendecomposition and the two Marchenko-Pastur edges

The correlation PCA puts 13 eigenvalues above its edge and the covariance PCA
puts 16 above its own. That is not a contradiction and it is not two estimates
of one quantity: the two matrices have different units and different edges.

INPUT: `data/processed/returns.parquet`, `data/processed/sectors.parquet`.
OUTPUT: none, printed and asserted only.

In [6]:
returns = pd.read_parquet(DATA / "processed" / "returns.parquet")
sectors = pd.read_parquet(DATA / "processed" / "sectors.parquet")
wide = st.clean_wide(returns)
wide = wide[[c for c in wide.columns if c in set(sectors["ticker"])]]
as_of = wide.index.max()

# a small sub-universe so every step below can be checked by eye: the last 252
# sessions across the first 60 sector-mapped names, so N is about 60 and T is
# about 252 rather than the other way round
small = wide.iloc[-252:, :60].dropna(axis=1)
z, _, _ = st.standardize(small)
n_days, n_names = small.shape
correlation = z.T @ z / n_days
covariance = np.cov(z, rowvar=False, ddof=0)
print(f"hand sub-universe: N {n_names}, T {n_days}, N/T {n_names / n_days:.4f}")

# the eigendecomposition of the correlation matrix, done twice: numpy and the
# module that produced the stored artifact, which must agree
values_hand, vectors_hand = np.linalg.eigh(correlation)
order = np.argsort(values_hand)[::-1]
values_hand, vectors_hand = values_hand[order], vectors_hand[:, order]
fit = st.fit(small)
reconstructed = vectors_hand @ np.diag(values_hand) @ vectors_hand.T
print(f"reconstruction error against the sample correlation matrix: "
      f"{np.abs(reconstructed - correlation).max():.3e}")
assert np.abs(reconstructed - correlation).max() < 1e-12
assert np.allclose(values_hand, np.sort(fit.eigenvalues)[::-1][:n_names], atol=1e-8)
print(f"largest hand eigenvalue {values_hand[0]:.6f} against the module's "
      f"{float(np.max(fit.eigenvalues)):.6f}")

# the edge in correlation units, and the same edge in variance units: the
# covariance spectrum is the correlation spectrum scaled by the mean variance
edge_correlation = st.mp_edge(n_names, n_days)
mean_variance = float(np.mean(np.diag(covariance)))
edge_covariance = mean_variance * edge_correlation
values_covariance = np.sort(np.linalg.eigvalsh(covariance))[::-1]
print(f"edge (correlation units)     {edge_correlation:.6f}")
print(f"mean variance                {mean_variance:.8f}")
print(f"edge (variance units)        {edge_covariance:.8f}")
print(f"above the correlation edge:  {int(np.sum(values_hand > edge_correlation))}")
print(f"above the covariance edge:   {int(np.sum(values_covariance > edge_covariance))}")
assert np.isclose(edge_correlation, (1 + np.sqrt(n_names / n_days)) ** 2)
assert np.isclose(edge_covariance / edge_correlation, mean_variance)

hand sub-universe: N 59, T 252, N/T 0.2341
reconstruction error against the sample correlation matrix: 2.026e-15
largest hand eigenvalue 9.374242 against the module's 9.374242
edge (correlation units)     2.201860
mean variance                0.99603175
edge (variance units)        2.19312284
above the correlation edge:  5
above the covariance edge:   5


In [7]:
# the same two counts on the model universe the sprint actually fitted, read
# from the registry rather than recomputed, and the reason they differ
pca_v1 = REGISTRY["models"]["PCA-v1"]["parameters"]
pca_v1c = REGISTRY["models"]["PCA-v1c"]["parameters"]
print(f"PCA-v1  correlation spectrum: N {pca_v1['n_names']}, T {pca_v1['n_days']}, "
      f"edge {pca_v1['mp_edge']:.4f}, count {pca_v1['n_factors_mp']}")
print(f"PCA-v1c covariance spectrum:  N {pca_v1c['n_names']}, T {pca_v1c['n_days']}, "
      f"edge {pca_v1c['mp_edge_covariance']:.6f}, count {pca_v1c['n_factors']}")
print(f"the covariance edge is the correlation edge scaled by the mean variance: "
      f"{pca_v1c['mp_edge_covariance'] / pca_v1['mp_edge']:.8f} against a mean "
      f"variance of {pca_v1c['mean_variance']:.8f}")
assert pca_v1["n_names"] == pca_v1c["n_names"] and pca_v1["n_days"] == pca_v1c["n_days"]
assert np.isclose(
    pca_v1c["mp_edge_covariance"] / pca_v1["mp_edge"], pca_v1c["mean_variance"]
)
print()
print("same N and T, two matrices in different units, two edges and therefore")
print("two counts. F4.2 governs the correlation count; F4.7 stores the other.")

PCA-v1  correlation spectrum: N 494, T 504, edge 3.9602, count 13
PCA-v1c covariance spectrum:  N 494, T 504, edge 0.002032, count 16
the covariance edge is the correlation edge scaled by the mean variance: 0.00051319 against a mean variance of 0.00051319

same N and T, two matrices in different units, two edges and therefore
two counts. F4.2 governs the correlation count; F4.7 stores the other.


## 4. By hand: the Ledoit-Wolf shrinkage intensity, with and without the 1/T

The defect this sprint found in a published estimator: without the `1/T` the
ratio is O(T) too large, clips at one in every window, and the estimator is
silently replaced by its constant-correlation target.

INPUT: `data/processed/returns.parquet`.
OUTPUT: none, printed and asserted only.

In [8]:
sample = cov.sample_cov(z)
target, mean_rho = cov.constant_correlation_target(sample)
squared = z**2
second_moment = squared.T @ squared / n_days
pi_hat = float(np.sum(second_moment - sample**2))
rho_hat = float(np.sum((second_moment - sample) * (target - sample)))
gamma_hat = float(np.sum((target - sample) ** 2))
corrected = float(np.clip((pi_hat - rho_hat) / (gamma_hat * n_days), 0.0, 1.0))
uncorrected = float(np.clip((pi_hat - rho_hat) / gamma_hat, 0.0, 1.0))
print(f"pi (sampling variance of the entries) {pi_hat:.6e}")
print(f"rho (cross term, returned not assumed) {rho_hat:.6e}")
print(f"gamma (distance to the target)        {gamma_hat:.6e}")
print(f"T                                     {n_days}")
print(f"ratio before the 1/T                  {(pi_hat - rho_hat) / gamma_hat:.4f}")
print(f"intensity without the correction      {uncorrected:.4f}")
print(f"intensity with the 1/T                {corrected:.4f}")
parts = cov.shrinkage_intensity(z, sample, target)
print(f"the module agrees:                    {parts['delta']:.4f}")
assert np.isclose(corrected, parts["delta"])
assert uncorrected == 1.0, "the uncorrected ratio should clip at one"
assert 0.0 <= corrected <= 1.0
shrunk, diagnostics = cov.ledoit_wolf(z)
assert np.isclose(diagnostics["delta"], corrected)
print(f"mean off-diagonal correlation of the target {mean_rho:.6f}")

pi (sampling variance of the entries) 4.359231e+03
rho (cross term, returned not assumed) 1.191678e+01
gamma (distance to the target)        9.837993e+01
T                                     252
ratio before the 1/T                  44.1890
intensity without the correction      1.0000
intensity with the 1/T                0.1754
the module agrees:                    0.1754
mean off-diagonal correlation of the target 0.111960


## 5. By hand: one out-of-sample minimum-variance portfolio, reconciled

One stored window, one estimator, the closed-form minimum-variance weights, the
volatility the estimator predicted and the volatility that happened. Both are
reconciled against the row the horse race stored for that date.

INPUT: `data/eval/cov_horse_race.parquet`, `data/processed/returns.parquet`.
OUTPUT: none, printed and asserted only.

In [9]:
race = pd.read_parquet(DATA / "eval" / "cov_horse_race.parquet")
window_end = pd.Timestamp(race["date"].max())
row = race.loc[
    (race["date"] == window_end) & (race["estimator"] == "ledoit_wolf")
].iloc[0]

# the horse race's own convention, rebuilt here: the 504 sessions up to the
# rebalance date, the next 21 sessions held with no re-estimation, and only the
# names complete in both windows
panel = st.clean_wide(returns)
panel = panel[[c for c in panel.columns if c in set(sectors["ticker"])]]
train = panel.loc[:window_end].iloc[-504:]
forward = panel.loc[window_end:].iloc[1:22]
names = [c for c in panel.columns if train[c].notna().all() and forward[c].notna().all()]
training = train[names].to_numpy(dtype=float)
realized_matrix = forward[names].to_numpy(dtype=float)

estimated = cov.estimator_from_window(training, "ledoit_wolf")
weights = cov.min_variance_weights(estimated.matrix)
predicted = cov.portfolio_vol(weights, estimated.matrix) * np.sqrt(252.0)
realized = cov.realized_vol(weights, realized_matrix)
print(f"window ends {window_end.date()}, {len(names)} names, {realized_matrix.shape[0]} forward days")
print(f"weights sum to {weights.sum():.10f}, gross {np.abs(weights).sum():.4f}")
print(f"predicted volatility {predicted:.6f}, stored {float(row['predicted_vol']):.6f}")
print(f"realized volatility  {realized:.6f}, stored {float(row['realized_vol']):.6f}")
print(f"condition number {estimated.condition_number:.4e}, stored {float(row['condition_number']):.4e}")
assert names and len(names) == int(row["n_names"])
assert abs(weights.sum() - 1.0) < 1e-10
assert abs(predicted - float(row["predicted_vol"])) < 1e-6
assert abs(realized - float(row["realized_vol"])) < 1e-6
assert np.isclose(estimated.condition_number, float(row["condition_number"]))
print()
print("the closed form reproduces the stored row to machine precision, so the")
print("race's numbers come from the construction this cell performs by hand.")

window ends 2026-07-31, 494 names, 21 forward days
weights sum to 1.0000000000, gross 5.6897
predicted volatility 0.062625, stored 0.062625
realized volatility  0.092071, stored 0.092071
condition number 2.8089e+03, stored 2.8089e+03

the closed form reproduces the stored row to machine precision, so the
race's numbers come from the construction this cell performs by hand.


## 6. F4.1 and F4.4 as specification conflicts, not model defects

F4.1 asks for the first principal component against the market and was written
for a PC1 that is a market portfolio. F4.4 asks PCA to beat a daily-refitted
XS-v1 on held-out cross-sectional variance. Both fail, and no third PCA was
constructed to make either pass.

INPUT: `data/eval/e4_f41_pc1_correlations.parquet`,
`data/eval/e4_f44_held_out.parquet`, `data/models/registry.json`.
OUTPUT: none, printed and asserted only.

In [10]:
pc1 = pd.read_parquet(DATA / "eval" / "e4_f41_pc1_correlations.parquet").set_index("variant")
print(pc1[["pc1_vs_market", "pc1_vs_equal_weight"]].round(6).to_string())
print()
print("what F4.1 assumed: one PC1, which would be the market portfolio")
print("what was built: two PC1s, one from the correlation matrix (an")
print("  equal-weight object) and one from the covariance matrix (a cap-weighted")
print("  object). The market factor itself correlates with the equal-weight mean")
print(f"  at {float(pc1.loc['market factor vs equal weight', 'pc1_vs_equal_weight']):.6f},")
print("  so no PC1 of either kind had the required correlation available.")
assert float(pc1.loc["PCA-v1 correlation", "pc1_vs_equal_weight"]) > 0.95
assert float(pc1.loc["PCA-v1 correlation", "pc1_vs_market"]) < 0.95
assert float(pc1.loc["PCA-v1c covariance", "pc1_vs_market"]) < 0.95
assert sorted(REGISTRY["models"]) == ["PCA-v1", "PCA-v1c", "TS-v1", "XS-v1"]
print()
print("F4.4, the comparison it names, on the same 503 held-out days:")
print(held_out["mean_r_squared"].round(6).to_string())
assert float(held_out.loc["(iv) PCA rolling refit", "mean_r_squared"]) < float(
    held_out.loc["(i) XS-v1 daily refit, as stored", "mean_r_squared"]
)
print()
print("neither threshold is reworded. The row that matters for E5 is (v).")

                               pc1_vs_market  pc1_vs_equal_weight
variant                                                          
PCA-v1 correlation                  0.797019             0.989144
PCA-v1c covariance                  0.930599             0.943251
market factor vs equal weight       1.000000             0.855646
PCA-v1 full sample                  0.945035                  NaN

what F4.1 assumed: one PC1, which would be the market portfolio
what was built: two PC1s, one from the correlation matrix (an
  equal-weight object) and one from the covariance matrix (a cap-weighted
  object). The market factor itself correlates with the equal-weight mean
  at 0.855646,
  so no PC1 of either kind had the required correlation available.

F4.4, the comparison it names, on the same 503 held-out days:
row
(i) XS-v1 daily refit, as stored     0.385727
(ii) XS-v1 descriptors frozen        0.253997
(iii) PCA frozen k=MP                0.271055
(iii) PCA frozen k=17                0.305

## 7. The survivor restriction: a data problem no model fixes

The XS-v1 universe is the 502 names with a sector mapping; the panel has 825.
No point-in-time sector source is reachable (Task 0b).

INPUT: `data/eval/xs_survivor_measurement.parquet`,
`data/eval/xs_survivor_universe_summary.parquet`,
`data/eval/xs_survivor_excluded_names.parquet`.
OUTPUT: none, printed and asserted only.

In [11]:
survivor = pd.read_parquet(DATA / "eval" / "xs_survivor_measurement.parquet").set_index("factor")
summary = pd.read_parquet(DATA / "eval" / "xs_survivor_universe_summary.parquet").set_index("universe")
excluded = pd.read_parquet(DATA / "eval" / "xs_survivor_excluded_names.parquet").set_index("group")
print(survivor[["correlation_panel_vs_mapped", "premium_panel", "premium_mapped"]].round(6).to_string())
print()
print(summary.round(8).to_string())
print()
print(excluded[['names', 'annualized_vol', 'differential_annualized']].round(6).to_string())
print()
print("The restriction is concentrated where the excluded names live: size and")
print("liquidity collapse across universes while the other styles hold. XS-v1's")
print("premia, R squared and specific variance are all flattered by it, and this")
print("is a data problem: only a point-in-time sector source closes it.")
assert survivor.loc["size", "correlation_panel_vs_mapped"] < 0.9
assert survivor.loc["market", "correlation_panel_vs_mapped"] > 0.99
assert summary.loc["mapped_502", "mean_r_squared"] > summary.loc["panel_825", "mean_r_squared"]
assert summary.loc["panel_825", "mean_names"] > summary.loc["mapped_502", "mean_names"]

           correlation_panel_vs_mapped  premium_panel  premium_mapped
factor                                                               
market                        0.997009       0.000201        0.000146
size                          0.577780      -0.000032       -0.000173
beta                          0.988961      -0.000259       -0.000255
momentum                      0.982820       0.000309        0.000276
reversal                      0.980173      -0.000519       -0.000466
resid_vol                     0.893133       0.000102        0.000120
liquidity                     0.625560      -0.000012       -0.000030

            mean_r_squared  mean_specific_variance  dates  mean_names
universe                                                             
panel_825         0.133530                0.000292    189  569.153439
mapped_502        0.142526                0.000229    189  467.941799

                              names  annualized_vol  differential_annualized
group      

## 8. New diagnostic: the effective sample size behind the EWMA row

EWMA is the only estimator worse than the sample covariance (median realized
vol 0.298494 against 0.280404) and its median condition number is 1.66e7, the
largest in the race. Both facts are what near rank deficiency looks like. An
exponentially weighted covariance with half-life h behaves like a sample of
`(1 + beta) / (1 - beta)` days, with `beta = 0.5^(1/h)`. If that effective
sample size is below N, the matrix is being estimated from fewer effective
observations than it has dimensions, and the row is explained rather than
merely observed.

INPUT: `data/eval/cov_horse_race.parquet`, `efb/cov.py` for the half-life.
OUTPUT: none, printed and recorded in the memo as one sentence.

In [12]:
half_life = int(cov.EWMA_HALF_LIFE)
beta = 0.5 ** (1.0 / half_life)
effective_t = (1.0 + beta) / (1.0 - beta)
n_ewma = int(race.loc[race["estimator"] == "ewma", "n_names"].median())
median_condition = float(
    race.loc[race["estimator"] == "ewma", "condition_number"].median()
)
print(f"EWMA half-life {half_life} days, beta {beta:.6f}")
print(f"implied effective sample size {effective_t:.2f} days")
print(f"median N on those windows {n_ewma}")
print(f"median condition number {median_condition:.4e}")
print(f"effective T below N: {effective_t < n_ewma}")
assert effective_t < n_ewma, "the effective sample size is not below N"
ratio = effective_t / n_ewma
print(f"effective T is {ratio:.4f} of N: the matrix is estimated from about "
      f"{ratio * 100:.1f} percent of the observations its dimension requires.")
print()
print("RECORDED: the EWMA row is rank deficient by construction rather than a")
print("tuning failure; the estimator is not recommended for any use.")

EWMA half-life 63 days, beta 0.989058
implied effective sample size 181.78 days
median N on those windows 466
median condition number 1.6581e+07
effective T below N: True
effective T is 0.3901 of N: the matrix is estimated from about 39.0 percent of the observations its dimension requires.

RECORDED: the EWMA row is rank deficient by construction rather than a
tuning failure; the estimator is not recommended for any use.


## 9. The D3 panel-to-column map, and the non-empty guard

D3 reads parquet only and never fits a model. Every panel builder goes through
`_require`, which raises on an empty read, so a panel whose artifact is missing
fails loudly instead of drawing an empty chart.

INPUT: `data/models/PCA-v1/*`, `data/models/PCA-v1c/*`, `data/eval/*`.
OUTPUT: the panel-to-column map, printed and asserted.

In [13]:
from dashboard.tabs import d03_covariance as d3  # noqa: E402

PANELS = {
    "eigenvalue_panel": d3.eigenvalue_panel,
    "explained_variance_panel": d3.explained_variance_panel,
    "factor_correlation_panel": d3.factor_correlation_panel,
    "estimator_table_panel": d3.estimator_table_panel,
    "halflife_panel": d3.halflife_panel,
    "residual_direction_panel": d3.residual_direction_panel,
}
for name, builder in PANELS.items():
    frame = builder()
    assert not frame.empty, name
    print(f"{name}  {frame.shape[0]} rows x {frame.shape[1]} cols")
    print(f"    columns: {', '.join(map(str, frame.columns))}")

guard_fired = 0
for name, builder in PANELS.items():
    try:
        d3._require(pd.DataFrame(), name)
    except ValueError:
        guard_fired += 1
assert guard_fired == len(PANELS), "a panel builder would draw an empty chart"
print(f"\nthe non-empty guard fired on {guard_fired} of {len(PANELS)} panels")

eigenvalue_panel  1093 rows x 10 cols
    columns: model, index, eigenvalue, explained_share, cumulative_share, mp_edge, n_names, n_days, n_over_t, above_edge
explained_variance_panel  494 rows x 4 cols
    columns: index, explained_share, cumulative_share, mp_edge
factor_correlation_panel  13 rows x 18 cols
    columns: beta, liquidity, market, momentum, resid_vol, reversal, sector_10, sector_15, sector_20, sector_25, sector_30, sector_35, sector_40, sector_45, sector_50, sector_55, sector_60, size
estimator_table_panel  9 rows x 6 cols
    columns: median_realized_vol, windows_won, median_condition_number, parameters, median_ratio_to_sample, windows
halflife_panel  9 rows x 4 cols
    columns: predicted_vol, realized_vol, bias, momentum_share
residual_direction_panel  3 rows x 8 cols
    columns: largest_eigenvalue, mp_edge, n_above_edge, effective_directions, top1_share, top3_share, top5_share, top10_share

the non-empty guard fired on 6 of 6 panels


## 10. Evidence for the deliverable, in citation order

`docs/research/E4_covariance_memo.md` cites these tables. Each is read from an
artifact here, in the order the memo uses them, so a reader can walk the memo
and this notebook side by side.

INPUT: `data/eval/cov_horse_race.parquet` and the criteria block.
OUTPUT: none, printed and asserted only.

In [14]:
pivot = race.pivot(index="date", columns="estimator", values="realized_vol")
table = pd.DataFrame(
    {
        "median_realized_vol": pivot.median(),
        "windows_won": (pivot.rank(axis=1, method="min") == 1).sum(),
        "median_condition_number": race.groupby("estimator")["condition_number"].median(),
        "fitted_quantities": race.groupby("estimator")["parameter_count"].median(),
    }
).sort_values("median_realized_vol")
print(table.round(6).to_string())
assert table.index[0] == "clip"
assert table.loc["sample", "windows_won"] == 0
assert table.loc["ewma", "windows_won"] == 0
assert table.loc["xs_v1", "windows_won"] == table["windows_won"].max()
assert np.isclose(table.loc["sample", "median_realized_vol"], STORED["F4.3"]["median_realized_vol"]["sample"])
print()
print("the memo's headline numbers, checked against the results file:")
for name, value in headlines.items():
    stored_value = STORED[name]
    assert value is not None and stored_value is not None
    print(f"  {name}  {value}")

                      median_realized_vol  windows_won  median_condition_number  fitted_quantities
estimator                                                                                         
clip                             0.086042           24             1.239682e+03              475.0
pca_v1                           0.086674           33             3.062064e+03              476.0
pca_v1c                          0.087318           29             1.928590e+03              477.0
xs_v1                            0.088007           59             2.716328e+03              484.0
ledoit_wolf                      0.094102           27             2.784871e+03                2.0
ts_v1                            0.150948            3             1.296088e+03              932.0
constant_correlation             0.160254            0             9.210483e+02              467.0
sample                           0.280404            0             2.650148e+06           108811.0
ewma      

## 11. What E5 inherits

Four registered versions, none champion, the champion rule untouched. Two
failing criteria with their mechanisms recorded. One clear candidate.

INPUT: `data/models/registry.json`, `sprints/E4/RESULTS.json`.
OUTPUT: none, printed and asserted only.

In [15]:
for name, entry in sorted(REGISTRY["models"].items()):
    print(f"{name:9s} family {entry['family']:11s} champion {entry['champion']} "
          f"eligible {entry['eligible_for_champion']}")
print()
print(f"champion declared: {registry.champion(REGISTRY)}")
assert registry.champion(REGISTRY) is None
assert REGISTRY["champion_rule"] == registry.DEFAULT_CHAMPION_RULE
assert sorted(registry.eligible(REGISTRY)) == ["PCA-v1", "PCA-v1c", "XS-v1"]
print(f"the champion rule, unchanged: {REGISTRY['champion_rule']}")
print()
for name in ("F4.1", "F4.4"):
    print(f"{name} carried into E5 as {VERDICTS[name]}: "
          f"{RESULTS['criteria'][name]['note'].split('.')[0]}.")
print()
print("leading candidate: XS-v2, XS-v1 with a residual covariance, on the")
print("strength of the held-out gain from three residual principal components.")

PCA-v1    family statistical champion False eligible True
PCA-v1c   family statistical champion False eligible True
TS-v1     family timeseries  champion False eligible False
XS-v1     family fundamental champion False eligible True

champion declared: None
the champion rule, unchanged: min mean |bias-1| across portfolio families; ties to fewer parameters; a champion must be refreshable daily from EFB's own data

F4.1 carried into E5 as fail: Fails on both variants.
F4.4 carried into E5 as fail: Fails by 7.

leading candidate: XS-v2, XS-v1 with a residual covariance, on the
strength of the held-out gain from three residual principal components.


## 12. Credit port note: what changes when the cross-section is bonds

No data here. This is the arithmetic of the same model applied to a corporate
bond universe, and the places where this sprint's findings should be larger.

INPUT: `data/models/registry.json` for N and T. OUTPUT: printed only.

In [16]:
n_stocks = int(pca_v1["n_names"])
t_days = int(pca_v1["n_days"])
for n_bonds in (300, 500):
    edge_eq = (1 + np.sqrt(n_stocks / t_days)) ** 2
    edge_bonds = (1 + np.sqrt(n_bonds / t_days)) ** 2
    print(f"N {n_bonds} bonds: N/T {n_bonds / t_days:.4f}, edge {edge_bonds:.4f} "
          f"against {edge_eq:.4f} for {n_stocks} stocks")
print()
print("With a few hundred bonds the edge sits closer to the equities case, so the")
print("factor count does not explode on N alone: the count is driven by the shared")
print("structure in the panel, not by the dimension.")
print("Residual co-movement in credit is issuer-level and mechanical: one issuer's")
print("bonds share a recovery, a covenant set and a capital structure, so their")
print("specific returns are correlated by construction, not by information. A")
print("diagonal specific-risk model on bonds therefore understates risk for the")
print("same reason it does for a concentrated equity book, and more sharply: the")
print("missing covariance is largest exactly between instruments that any sensible")
print("portfolio holds together. Ranking is the second difference: with 500 stocks")
print("the residual directions are interpretable and unstable; with bonds they are")
print("issuer blocks, which are stable and namable, so a residual covariance is")
print("easier to justify there than here.")

N 300 bonds: N/T 0.5952, edge 3.1383 against 3.9602 for 494 stocks
N 500 bonds: N/T 0.9921, edge 3.9841 against 3.9602 for 494 stocks

With a few hundred bonds the edge sits closer to the equities case, so the
factor count does not explode on N alone: the count is driven by the shared
structure in the panel, not by the dimension.
Residual co-movement in credit is issuer-level and mechanical: one issuer's
bonds share a recovery, a covenant set and a capital structure, so their
specific returns are correlated by construction, not by information. A
diagonal specific-risk model on bonds therefore understates risk for the
same reason it does for a concentrated equity book, and more sharply: the
missing covariance is largest exactly between instruments that any sensible
portfolio holds together. Ranking is the second difference: with 500 stocks
the residual directions are interpretable and unstable; with bonds they are
issuer blocks, which are stable and namable, so a residual covariance is


## 13. Closing checklist

The last cell prints the checklist and scans this file's own code cells for any
stored number appearing as a literal. A stored value may be printed and
asserted, never typed.

In [17]:
def numeric_leaves(node):
    out = []
    if isinstance(node, dict):
        for value in node.values():
            out.extend(numeric_leaves(value))
    elif isinstance(node, list):
        for value in node:
            out.extend(numeric_leaves(value))
    elif isinstance(node, float):
        out.append(node)
    return out


notebook = json.loads((ROOT / "notebooks" / "E4_walkthrough.ipynb").read_text())
source = "\n".join(
    "".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code"
)
checked = numeric_leaves(RESULTS["criteria"]) + numeric_leaves(REGISTRY["models"])
offenders = sorted(
    {
        text
        for value in checked
        for text in (f"{value:.6f}", f"{value:.4f}")
        if len(text) > 6 and text in source
    }
)
assert not offenders, f"stored values typed into a code cell: {offenders}"
print("closing checklist")
print(f"  criteria evaluated:        {len(RESULTS['criteria'])}")
print(f"  verdicts:                  {sorted(set(VERDICTS.values()))}")
print(f"  data hash asserted:        {RESULTS['data_hash']}")
print(f"  artifacts hashed:          {len(evaluate.e4_artifacts(DATA))}")
print(f"  stored values scanned:     {len(checked)}")
print(f"  literals found in cells:   {len(offenders)}")
print("  the residual structure, the two specification conflicts, the survivor")
print("  restriction, the EWMA reading, the D3 map and the E5 handover are all")
print("  asserted above from stored artifacts.")

closing checklist
  criteria evaluated:        7
  verdicts:                  ['fail', 'pass']
  data hash asserted:        ff4e6b2d0002ab65e7e905e0908f57258236419454cfbb904b8cc85a47679018
  artifacts hashed:          16
  stored values scanned:     145
  literals found in cells:   0
  the residual structure, the two specification conflicts, the survivor
  restriction, the EWMA reading, the D3 map and the E5 handover are all
  asserted above from stored artifacts.
